# CASB: training-only family selection and novelty audit

This is a retrospective follow-up, with the selector specified on 2026-10-20 after the original benchmark was inspected. It is NOT a new untouched holdout study. Existing test-best family scores are an oracle reference, not the proposed selector score.

For each outer training fold, seed, and classifier: maximize the saved inner-validation (G-mean + F1)/2 over CASB, CASB-relaxed, CASB-edited and ratios 0.5, 0.75, 1.0. Ties choose the smaller ratio, then CASB, relaxed, edited in that order. The selected configuration was already refitted on the full outer training fold in the original run; its saved outer-test result is reused exactly. No test metrics enter selection.

Search budgets differ: CASB selector uses 9 inner fits; each fixed baseline uses 3. We additionally report a training-only baseline-family selector over SMOTE, BSMOTE, ROS, RUS (12 inner fits). This is not a matched-budget comparison. No classifier hyperparameters are searched. All 44 datasets are included; related dataset variants and reused folds prevent treating all observations as independent replications. Two Abalone datasets use one-hot encoding, where interpolation can yield fractional dummy values.


In [1]:
import sys, os, json, hashlib, copy
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import select_validation_candidate, choose_casb_variant, data_splits, evaluate_fit
from threadpoolctl import threadpool_limits
variants=('casb','casb_relaxed','casb_edited')
core=('smote','bsmote','ros','rus')
rows=[]; notebook_hashes={}
paths=sorted(Path('.').glob('[0-9][0-9]_*_CASB.ipynb'))
assert len(paths)==44
for path in paths:
    raw=path.read_bytes(); notebook_hashes[str(path)]=hashlib.sha256(raw).hexdigest()
    nb=json.loads(raw); meta=nb['metadata']['benchmark']; records=meta['records']
    assert meta['executed_complete'] and len(records)==180
    assert all('error' not in r for r in records)
    rows.extend(records)
frame=pd.DataFrame(rows)
keys=['dataset','classifier','seed','fold']
assert not frame.duplicated(keys+['method']).any()
assert frame.groupby(keys).size().eq(9).all()
assert frame.dataset.nunique()==44 and len(frame)==7920
print('Verified complete source records:',len(frame))


Verified complete source records: 7920


In [2]:
# The selector receives only validation dictionaries, not outer-test results.
selected=[]; decisions=[]
for key, group in frame.groupby(keys):
    indexed=group.set_index('method')
    for name,methods in [('casb_selected',variants),('baseline_selected',core)]:
        validation={m:indexed.loc[m,'validation'] for m in methods}
        assert all(len(v)==3 for v in validation.values())
        decision=select_validation_candidate(validation,methods)
        chosen=indexed.loc[decision['method']].to_dict()
        assert chosen['ratio']==decision['ratio']
        decisions.append(dict(zip(keys,key),selector=name,**decision))
        chosen.update(method=name,selected_variant=decision['method'])
        selected.append(chosen)
selected_frame=pd.DataFrame(selected)
assert len(selected_frame)==1760
all_scores=pd.concat([frame,selected_frame],ignore_index=True)
all_scores['composite']=(all_scores.gmean+all_scores.f1)/2
metrics=['gmean','f1','precision','recall','ba','ap','auc','composite']
means=all_scores.groupby(['dataset','classifier','method'])[metrics].mean()
ds=means.groupby(['dataset','method']).mean()
aggregate=ds.groupby('method').mean().sort_values('composite',ascending=False)
display(Markdown('## Equal dataset / classifier averages (%)'))
display((100*aggregate).round(3))
decisions=pd.DataFrame(decisions)
display(Markdown('## Training-only variant choices across 880 outer settings'))
display(pd.crosstab(decisions.selector,decisions.method))


## Equal dataset / classifier averages (%)

,gmean,f1,precision,recall,ba,ap,auc,composite
method,,,,,,,,
casb_selected,83.066,68.001,63.916,78.500,85.007,70.987,90.369,75.534
casb_relaxed,82.786,67.834,64.135,77.369,84.784,71.707,90.654,75.310
baseline_selected,83.279,67.241,62.870,79.016,84.966,71.265,90.657,75.260
smote,83.027,67.491,63.613,77.961,84.850,70.953,90.544,75.259
casb_edited,83.799,66.590,61.076,80.499,85.288,69.819,90.516,75.195
casb,82.233,68.145,64.885,76.659,84.658,71.433,90.210,75.189
ros,83.001,67.249,62.985,77.767,84.923,70.338,90.016,75.125
smoteenn,83.968,65.718,60.331,81.282,85.289,68.054,90.069,74.843
bsmote,81.768,67.222,62.555,76.970,84.377,70.129,90.077,74.495


## Training-only variant choices across 880 outer settings

method,bsmote,casb,casb_edited,casb_relaxed,ros,rus,smote
selector,,,,,,,
baseline_selected,190,0,0,0,139,178,373
casb_selected,0,512,165,203,0,0,0


In [3]:
def compare(table,candidate='casb_selected'):
    chosen=table.xs(candidate,level='method')
    outcomes=[]
    for baseline in list(core)+['baseline_selected']:
        delta=chosen-table.xs(baseline,level='method')
        win=int((delta.composite>1e-12).sum()); loss=int((delta.composite < -1e-12).sum())
        outcomes.append(dict(baseline=baseline,wins=win,ties=len(delta)-win-loss,losses=loss,
            composite_delta_pp=100*delta.composite.mean()))
    display(pd.DataFrame(outcomes).set_index('baseline').round(3))
    joint=[d for d in chosen.index if all(chosen.loc[d,'composite']>table.loc[(d,b),'composite']+1e-12 for b in core)]
    print('Training-only selector beats all four:',len(joint),'/',len(chosen))
    print('Winning datasets:',joint)
    return outcomes,joint
outcomes,joint=compare(ds)
for clf in ['svm','knn']:
    display(Markdown('## '+clf.upper()+' (%)'))
    display((100*means.xs(clf,level='classifier').groupby('method')[metrics].mean()).round(3).sort_values('composite',ascending=False))
remaining=ds.loc[~ds.index.get_level_values('dataset').isin(['pima','haberman','yeast3','ecoli4','vowel0','segment0'])]
display(Markdown('## Remaining 38 datasets (%) ? previously inspected, not pristine holdouts'))
display((100*remaining.groupby('method').mean()).round(3).sort_values('composite',ascending=False))
remaining_outcomes,remaining_joint=compare(remaining)
display(Markdown('## Per-dataset comparison (%)'))
display((100*ds[['composite','gmean','f1']]).round(3).unstack('method'))


,wins,ties,losses,composite_delta_pp
baseline,,,,
smote,20,3,21,0.274
bsmote,24,3,17,1.039
ros,24,3,17,0.409
rus,31,1,12,3.018
baseline_selected,23,3,18,0.274


Training-only selector beats all four: 8 / 44
Winning datasets: ['abalone9-18', 'haberman', 'new-thyroid2', 'pima', 'shuttle-c0-vs-c4', 'vehicle1', 'vehicle2', 'yeast-1-2-8-9_vs_7']


## SVM (%)

,gmean,f1,precision,recall,ba,ap,auc,composite
method,,,,,,,,
casb_selected,82.540,69.452,66.809,77.689,84.957,74.028,91.540,75.996
ros,83.426,68.481,65.285,78.079,85.358,73.599,91.837,75.953
casb_relaxed,82.341,69.309,67.000,76.539,84.787,74.405,91.762,75.825
casb_edited,83.481,68.152,63.839,79.545,85.253,73.373,91.573,75.817
smote,82.556,68.804,66.125,77.016,84.781,73.845,91.695,75.680
smoteenn,83.944,67.159,62.954,80.277,85.337,72.735,91.226,75.552
baseline_selected,82.963,68.113,64.723,78.170,85.048,73.583,91.687,75.538
casb,81.531,69.471,67.910,75.841,84.613,74.350,91.683,75.501
bsmote,80.767,67.933,64.514,75.445,84.010,72.781,91.183,74.350


## KNN (%)

,gmean,f1,precision,recall,ba,ap,auc,composite
method,,,,,,,,
casb_selected,83.593,66.551,61.023,79.310,85.058,67.946,89.198,75.072
baseline_selected,83.595,66.368,61.016,79.863,84.884,68.948,89.626,74.981
casb,82.935,66.820,61.860,77.476,84.702,68.517,88.736,74.878
smote,83.499,66.179,61.101,78.905,84.919,68.061,89.393,74.839
casb_relaxed,83.231,66.360,61.269,78.198,84.781,69.009,89.545,74.795
bsmote,82.769,66.510,60.595,78.494,84.745,67.476,88.971,74.639
casb_edited,84.118,65.029,58.312,81.453,85.324,66.265,89.458,74.573
ros,82.576,66.018,60.686,77.455,84.488,67.078,88.195,74.297
smoteenn,83.991,64.276,57.707,82.287,85.241,63.372,88.912,74.134


... (2 outputs left out)

,wins,ties,losses,composite_delta_pp
baseline,,,,
smote,17,2,19,0.106
bsmote,19,2,17,0.964
ros,20,2,16,0.292
rus,27,1,10,3.244
baseline_selected,19,2,17,0.142


Training-only selector beats all four: 6 / 38
Winning datasets: ['abalone9-18', 'new-thyroid2', 'shuttle-c0-vs-c4', 'vehicle1', 'vehicle2', 'yeast-1-2-8-9_vs_7']


## Per-dataset comparison (%)

composite                                \
method                 baseline_selected   bsmote     casb casb_edited   
dataset                                                                  
abalone19                         32.540   14.274   27.033      30.395   
abalone9-18                       60.009   58.610   62.315      61.724   
ecoli-0-1-3-7_vs_2-6              65.918   66.576   62.977      59.936   
ecoli-0_vs_1                      96.003   96.162   96.064      96.207   
ecoli1                            82.727   82.939   82.936      83.316   
ecoli2                            90.224   87.052   89.603      90.608   
ecoli3                            73.027   72.641   72.591      71.818   
ecoli4                            83.943   84.326   84.101      87.311   
glass-0-1-2-3_vs_4-5-6            89.911   90.809   89.706      87.754   
glass-0-1-6_vs_2                  49.540   46.058   42.467      50.578   
glass-0-1-6_vs_5                  80.957   80.957   78.014      76.405   
glass0                            75.468   75.831   75.709      74.249   
glass1                            68.442   68.501   69.815      70.851   
glass2                            49.205   42.765   41.275      49.925   
glass4                            74.655   81.645   79.290      76.898   
glass5                            74.422   77.799   74.784      68.598   
glass6                            84.085   81.988   86.347      84.106   
haberman                          49.707   49.603   49.541      55.636   
iris0                            100.000  100.000  100.000     100.000   
new-thyroid1                      96.854   97.260   98.185      97.606   
new-thyroid2                      96.445   96.818   97.812      97.192   
page-blocks-1-3_vs_4              93.108   93.888   93.795      91.867   
page-blocks0                      86.033   86.153   86.282      86.615   
pima                              67.606   68.055   68.114      69.890   
segment0                          98.381   97.726   97.828      97.910   
shuttle-c0-vs-c4                  99.175   98.964   99.481      99.379   
shuttle-c2-vs-c4                  93.738   93.738   93.738      93.738   
vehicle0                          92.571   92.356   91.940      90.348   
vehicle1                          69.823   69.936   69.716      69.756   
vehicle2                          94.497   94.347   94.615      94.207   
vehicle3                          70.423   70.197   69.892      69.227   
vowel0                            99.851   99.851   99.851      99.851   
wisconsin                         96.102   96.207   95.984      95.922   
yeast-0-5-6-7-9_vs_4              63.222   63.015   64.211      61.758   
yeast-1-2-8-9_vs_7                37.214   40.162   43.727      42.353   
yeast-1-4-5-8_vs_7                37.311   40.604   35.320      35.954   
yeast-1_vs_7                      51.778   53.027   51.860      53.062   
yeast-2_vs_4                      81.241   81.768   81.306      81.577   
yeast-2_vs_8                      60.911   46.914   65.728      62.455   
yeast1                            64.316   62.937   64.929      64.910   
yeast3                            81.617   79.150   81.112      81.131   
yeast4                            55.454   51.816   54.147      53.166   
yeast5                            79.811   79.926   80.036      79.279   
yeast6                            63.164   64.414   64.156      63.110   

                                                                              \
method                 casb_relaxed casb_selected original      ros      rus   
dataset                                                                        
abalone19                    27.765        29.062    0.000   24.821   32.471   
abalone9-18                  61.929        61.483   23.019   60.836   55.920   
ecoli-0-1-3-7_vs_2-6         65.284        63.880   70.114   55.813   67.594   
ecoli-0_vs_1                 96.405        96.103   95.924   95.805   95

In [4]:
# Test deterministic tie rules, failed trials, and rejection of missing evidence.
trials={m:[{'ratio':r,'ba':.5} for r in [.5,.75,1.]] for m in variants}
assert select_validation_candidate(trials)==dict(method='casb',ratio=.5,validation_score=.5)
trial2=copy.deepcopy(trials);trial2['casb_relaxed'][1]['ba']=.8
assert select_validation_candidate(trial2)['method']=='casb_relaxed'
assert select_validation_candidate(dict(reversed(list(trial2.items()))))==select_validation_candidate(trial2)
try: select_validation_candidate({})
except ValueError: pass
else: raise AssertionError('Missing trials must fail')
trial3={m:[{'ratio':.5,'error':'test failure'}] for m in variants}
try: select_validation_candidate(trial3)
except ValueError: pass
else: raise AssertionError('All failed trials must fail')
# Independently rerun inner search and outer refit on numeric and mixed input.
replay=[]
with threadpool_limits(limits=1):
    for dataset,clf in [('pima','svm'),('abalone9-18','knn')]:
        seed=42
        fold,X,y,Xt,yt,_=next(data_splits(Path('All'),dataset,5,seed,allow_nominal=True))
        decision,validation=choose_casb_variant(X,y,clf,seed)
        saved=decisions[(decisions.dataset==dataset)&(decisions.classifier==clf)&(decisions.seed==seed)&(decisions.fold==fold)&(decisions.selector=='casb_selected')].iloc[0]
        assert decision['method']==saved['method'] and decision['ratio']==saved['ratio']
        assert abs(decision['validation_score']-saved.validation_score)<1e-12
        actual=evaluate_fit(X,y,Xt,yt,decision['method'],clf,seed,decision['ratio'])
        reference=selected_frame[(selected_frame.dataset==dataset)&(selected_frame.classifier==clf)&(selected_frame.seed==seed)&(selected_frame.fold==fold)&(selected_frame.method=='casb_selected')].iloc[0]
        assert all(abs(actual[m]-reference[m])<1e-12 for m in metrics if m!='composite')
        replay.append(dict(dataset=dataset,classifier=clf,**decision))
display(pd.DataFrame(replay))
print('Selector behavior tests and two independent inner-search + outer-refit checks passed.')


,dataset,classifier,method,ratio,validation_score
0,pima,svm,casb_relaxed,1.0,0.807718
1,abalone9-18,knn,casb_edited,1.0,0.475283


Selector behavior tests and two independent inner-search + outer-refit checks passed.


## Preliminary mechanism-level novelty audit

An exact-name web search for "CASB-SMOTE" and "CASB oversampling" did not surface a matching paper in this search. This is not proof that the mechanism is unpublished. Search scope is preliminary, not a systematic review.

| CASB component | Relevant prior work | Current assessment |
|---|---|---|
| Distinguishing safe/border/rare examples | [Napierala and Stefanowski](https://doi.org/10.1007/s10844-015-0368-1); [Skryjomski and Krawczyk, 2017](https://proceedings.mlr.press/v74/skryjomski17a.html) | The categorization and study of type-specific oversampling predate CASB. CASB's distance-based rare pool differs from a purely label-neighborhood definition; that difference needs evaluation. |
| Allocate synthesis across source types | [Local Neighborhood Encodings, 2024](https://link.springer.com/article/10.1007/s10994-024-06563-6) | LNE already adjusts resampling intensity by neighborhood type. CASB instead uses fixed mixture quotas, including a pool containing all minority samples. Exact equivalence is not established; type allocation alone is not a first. |
| Restrict interpolation near majority points | [Safe-Level-SMOTE, 2009](https://sci2s.ugr.es/keel/pdf/algorithm/congreso/2009-Bunkhumpornpat-LNCS.pdf); [Geometric SMOTE](https://arxiv.org/abs/1709.07377) | Safety-aware and geometric synthesis are established. CASB's particular distance-ratio step cap is the narrower rule to compare directly. |
| Synthesis followed by ENN | [imbalanced-learn combination methods](https://imbalanced-learn.org/stable/combine.html) | SMOTE+ENN is established. Majority-only, mode-vote cleaning must be matched in the baseline to isolate CASB generation. |
| Validation-based variant selection | [Cawley and Talbot, 2010](https://www.jmlr.org/beta/papers/v11/cawley10a.html) | Standard model selection; useful for valid evaluation, not a novel mechanism by itself. |

### What the implemented rules actually guarantee

The coverage pool contains every minority sample, but random source selection with replacement does **not** guarantee that every minority observation is used. The rare pool is determined by large third-neighbor distance (upper quartile), plus minority support, rather than an explicit rare-subcluster detector. Border and rare pools may overlap. Pool quotas therefore count branch draws, not disjoint population strata. Empty-pool fallback preserves the original branch's neighbor and cap behavior.

For a **capped** synthetic step z from source x, let d be its distance to the nearest observed majority point and r=||z-x||. If r <= c*d with c <= 1/2, then every observed majority point b satisfies ||z-b|| >= d-r >= r by the triangle inequality. This is a deterministic statement about observed training geometry, not the true class boundary or generalization. It does NOT apply to unrestricted safe-coverage steps or to CASB-relaxed/CASB-edited, which disable the cap. This elementary bound is not claimed as a new theorem.

### Claim supported at this stage

?We evaluate a fixed-mixture oversampling design combining broad minority-source eligibility, locally defined borderline and sparse-region source pools, and branch-specific interpolation constraints, with relaxed and cleaned variants.?

This describes the implemented approach without asserting a first-in-literature result. The exact combination is a **candidate incremental contribution** whose distinctness remains unverified. A win on a test dataset does not certify novelty; absence of an exact name does not certify it either.

### Required work before a stronger claim

- Matched-cleaner ablation: vanilla SMOTE followed by exactly the same majority-only 3-neighbor mode ENN as CASB-edited. Existing library SMOTEENN is not this matched control.
- Isolate each change: remove border pool, remove rare pool, use coverage only, disable the cap with original quotas, and change quotas without also disabling the cap. Existing CASB-versus-relaxed comparison changes two factors at once.
- Compare against relevant source-type and geometry-aware baselines, not only SMOTE/BSMOTE/ROS/RUS.
- Freeze the resulting procedure and evaluate on new dataset families. All 44 current datasets have already been inspected; label them exploratory/development evidence. Avoid treating related KEEL variants or repeated folds as independent evidence.
- Investigate duplicate feature vectors across supplied train/test folds and report sensitivity under group-aware splits. Record uncertainty without claiming significance from correlated fold scores.

No new algorithmic novelty, significant superiority, or untouched-holdout claim is made by this notebook. Original benchmark results are preserved.
